# Contoso Forge: interactive Spark to BigQuery
Experimental generated notebook. Classic and true Connect local are separate requested modes. Each run records the actual API mode and exact versions; local validation does not imply hosted Colab validation.
For a Spark-only gate, package with --scope spark; no GCP project or authentication is required. For Spark and BigQuery, set project/dataset/location in project.json before generation, create the dataset in BigQuery Sandbox, and package with --scope spark-and-bigquery.
On your computer, run python colab/work_order.py package --root . --run-id demo-001 --scope spark. Upload its work_package.zip below. For BigQuery loading, instead issue a Spark-and-BigQuery work order. The notebook reuses V1 CDC/SCD2/quality rules, writes Parquet and returns a Spark manifest before the optional warehouse step. Airflow cannot start this notebook unattended.
Each work order uses unique table names and WRITE_EMPTY loads. Sandbox tables expire according to Sandbox policy. Billing-enabled free allowances are not a guarantee of zero charges; query bytes and file size are capped by generated config.

The generated spark_config.json selects classic or connect-local. colab-native reuses compatible installed PySpark (including 4.0.4); it refuses unknown versions. Pinned replacement requires an explicit notebook choice. Preserve the returned Spark result even if BigQuery authentication or loading fails.

Optional upload path: use the Colab Files panel to upload the ZIP, then set WORK_PACKAGE_PATH in the upload cell to its /content path. Leave it None for the interactive upload widget.


In [ ]:
import os, sys, subprocess, json, io, zipfile, stat, uuid
from pathlib import Path


In [ ]:
from google.colab import files
WORK_PACKAGE_PATH = None  # Optional: '/content/work_package.zip' after Files-panel upload.
uploaded = {Path(WORK_PACKAGE_PATH).name: Path(WORK_PACKAGE_PATH).read_bytes()} if WORK_PACKAGE_PATH else files.upload()
assert len(uploaded) == 1, 'Upload exactly one work_package.zip'
root = Path('/content') / ('contoso_' + uuid.uuid4().hex)
root.mkdir()
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as package:
    names = set()
    assert sum(x.file_size for x in package.infolist()) <= 500_000_000, 'Package exceeds the small-lab limit'
    for item in package.infolist():
        target = (root / item.filename).resolve()
        assert target.is_relative_to(root) and target != root and '\\' not in item.filename, 'Unsafe ZIP path'
        assert item.filename not in names and not stat.S_ISLNK(item.external_attr >> 16), 'Unsafe ZIP member'
        names.add(item.filename)
    package.extractall(root)
os.chdir(root)
sys.path.insert(0, str(root / 'colab'))
from work_order import read_json, validate_order
order = read_json(root / 'colab/work_order.json')
validate_order(root, order)
print('Work order:', order['workOrderId'], 'Run:', order['runId'])
print('Dataset:', order['gcp']['projectId'] + '.' + order['gcp']['dataset'])


In [ ]:
spark_config = read_json(root / 'colab/spark_config.json')
print('Requested Spark settings:', json.dumps(spark_config, indent=2))
# Set True only when deliberately replacing installed PySpark under the pinned policy.
ALLOW_PINNED_VERSION_CHANGE = False
bootstrap_args = [sys.executable, 'colab/bootstrap_runtime.py', '--config', 'colab/spark_config.json']
if ALLOW_PINNED_VERSION_CHANGE:
    bootstrap_args.append('--allow-version-change')
subprocess.run(bootstrap_args, check=True)


In [ ]:
subprocess.run([sys.executable, 'colab/run_spark.py', '--root', '.', '--lake-root', 'lake', '--work-order', 'colab/work_order.json'], check=True)


In [ ]:
# Return Spark proof before any optional warehouse step.
subprocess.run([sys.executable, 'colab/work_order.py', 'spark-result', '--root', '.', '--work-order', 'colab/work_order.json', '--runtime', 'colab/spark_runtime.json', '--result', 'colab/spark_result_manifest.json'], check=True)
files.download(str(root / 'colab/spark_result_manifest.json'))
print(json.dumps(read_json(root / 'colab/spark_runtime.json'), indent=2))


In [ ]:
if order.get('executionScope', 'spark-and-bigquery') == 'spark-and-bigquery':
    from google.colab import auth
    auth.authenticate_user()
    # Uses the signed-in user's standard credentials. No credential is stored in the package.
else:
    print('Spark-only work order complete; warehouse step skipped.')


In [ ]:
if order.get('executionScope', 'spark-and-bigquery') == 'spark-and-bigquery':
    subprocess.run([sys.executable, 'gcp/bigquery_runtime.py', 'run', '--root', '.', '--silver-root', 'lake/silver', '--work-order', 'colab/work_order.json', '--result', 'colab/result_manifest.json'], check=True)
else:
    print('Spark-only work order complete; warehouse step skipped.')


In [ ]:
if order.get('executionScope', 'spark-and-bigquery') == 'spark-and-bigquery':
    files.download(str(root / 'colab/result_manifest.json'))
    print('Return this file to the matching Airflow run state directory, or reconcile locally with:')
    print('python colab/work_order.py reconcile --root . --work-order colab/work_order.json --result colab/result_manifest.json')
else:
    print('Spark-only work order complete; warehouse step skipped.')
